In [1]:
import h3
import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import io, requests

# Create hourly aggregated Dateframe
Create a new df with hourly timestamps in the range of the dataset and all unique H3 indices so we have a complete grid for analysis

In [2]:
# Loading all 23 raw columns of both files costs ~22 GB of Python strings
# Each set only ever feeds the levels below, so read just those columns and
# narrow the dtypes on the way in.
TS_FORMAT = '%m/%d/%Y %I:%M:%S %p'

BASE_COLS = [
    'Taxi ID', 'Trip Start Timestamp', 'Trip End Timestamp',
    'Trip Seconds', 'Trip Miles', 'Trip Total', 'Company',
    'Pickup Centroid Latitude', 'Pickup Centroid Longitude',
]
# taxi_data_ca feeds the community level; taxi_data_census feeds the h3 and census levels.
CA_COLS = BASE_COLS + ['Pickup Community Area', 'Dropoff Community Area']
CENSUS_COLS = BASE_COLS + ['Pickup Census Tract', 'Dropoff Census Tract',
                           'h3_index_pickup_7', 'h3_index_dropoff_7',
                           'h3_index_pickup_8', 'h3_index_dropoff_8']


def load_trips(path, columns):
    df = pd.read_parquet(path, columns=columns)

    # Floor to the hour once here, then drop the raw strings: 22 chars x 21M rows is the single
    # biggest block of memory in the notebook, and nothing downstream needs minute resolution.
    df['Pickup Hour']  = pd.to_datetime(df['Trip Start Timestamp'], format=TS_FORMAT).dt.floor('h')
    df['Dropoff Hour'] = pd.to_datetime(df['Trip End Timestamp'],   format=TS_FORMAT).dt.floor('h')
    df = df.drop(columns=['Trip Start Timestamp', 'Trip End Timestamp'])

    # ~3.4k taxis and 44 companies across 14M rows: store codes, not one string per row.
    for col in ('Taxi ID', 'Company'):
        df[col] = df[col].astype('category')

    return df


taxi_data_ca     = load_trips('../data/processed/taxi_data_processed_big.parquet',   CA_COLS)
taxi_data_census = load_trips('../data/processed/taxi_data_processed_small.parquet', CENSUS_COLS)

for name, df in (('community set', taxi_data_ca), ('census set', taxi_data_census)):
    print(f'{name:14s} {len(df):>10,} trips   {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

community set  14,494,804 trips   1.20 GB
census set      6,835,451 trips   1.20 GB


## Spatial Chicago Ground Truth
Build a combined lookup with h3_8 as the finest level, matched to h3_7, census tract (geoid10) and community area (commarea).

In [3]:
# Chicago census tract boundaries (2010) — download from the city data portal and cache locally
C_PATH = Path('..') / 'data' / 'chicago_census_comm.gpkg'

if C_PATH.exists():
    census_tracts = gpd.read_file(C_PATH)
else:
    print('Downloading Chicago census tract and community area boundaries...')
    url = 'https://data.cityofchicago.org/resource/74p9-q2aq.geojson?$limit=1000'
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    census_tracts = gpd.read_file(io.BytesIO(r.content))
    census_tracts.to_file(C_PATH, driver='GPKG')

    print(f'Saved {len(census_tracts)} census tracts to {C_PATH}')

all_census_comm = census_tracts.to_crs(epsg=4326)
print(f'Census tracts loaded: {len(all_census_comm)}')

Census tracts loaded: 801


In [4]:
# Dissolve the community areas into a single boundary for the whole city
chicago_boundary = all_census_comm.union_all()

# Finest level: all H3 res-8 cells covering Chicago (h3 v4 API replaces v3 polyfill)
h3_index_8 = sorted(h3.geo_to_cells(chicago_boundary, 8))
all_census_tracts = pd.DataFrame({'h3_index_8': h3_index_8})

# h3_7 is simply the res-7 parent of each res-8 cell
all_census_tracts['h3_index_7'] = all_census_tracts['h3_index_8'].apply(
    lambda c: h3.cell_to_parent(c, 7)
)

# Match census tract + community area via the location of each hex centroid
centroids = all_census_tracts['h3_index_8'].apply(h3.cell_to_latlng)
hex_points = gpd.GeoDataFrame(
    all_census_tracts,
    geometry=gpd.points_from_xy(
        centroids.apply(lambda x: x[1]),  # lng
        centroids.apply(lambda x: x[0]),  # lat
    ),
    crs='EPSG:4326',
)
# Perform a spatial join to match each hexagon centroid to the corresponding census tract and community area
all_census_tracts = gpd.sjoin(
    hex_points,
    all_census_comm[['geoid10', 'commarea', 'geometry']],
    how='left',
    predicate='within',
).drop(columns=['index_right', 'geometry'])

# One row per h3_8 cell, everything matched against it
all_census_tracts = pd.DataFrame(all_census_tracts).reset_index(drop=True)

print(
    f"{len(all_census_tracts)} h3_8 cells | "
    f"{all_census_tracts['h3_index_7'].nunique()} h3_7 parents | "
    f"{all_census_tracts['geoid10'].nunique()} census tracts | "
    f"{all_census_tracts['commarea'].nunique()} community areas"
)
all_census_tracts.head()

849 h3_8 cells | 153 h3_7 parents | 534 census tracts | 77 community areas


,h3_index_8,h3_index_7,geoid10,commarea
0,8826641901fffff,872664190ffffff,17031838800,51
1,8826641903fffff,872664190ffffff,17031838800,51
2,8826641905fffff,872664190ffffff,17031838800,51
3,8826641907fffff,872664190ffffff,17031838800,51
4,8826641909fffff,872664190ffffff,17031838800,51


In [5]:
# A geometric centre for every spatial unit, at every level.
#
# The aggregated PickupLatitude/PickupLongitude are *trip* averages, so they only exist where trips
# happened -- they are NaN for the ~97% of empty cells and therefore cannot locate a quiet hexagon.
# The unit's own centroid is defined regardless of demand, so it is the right fill value and also a
# usable spatial feature in its own right.
#
# h3 cells get their centre from the index itself; tracts and community areas get the centroid of
# their polygon, computed in a projected CRS (UTM 16N) so "centre" means centre on the ground rather
# than centre in degrees.
def polygon_centroids(gdf, key):
    dissolved = gdf[[key, 'geometry']].dissolve(by=key)          # community areas span many tracts
    centroids = dissolved.to_crs(epsg=26916).centroid.to_crs(epsg=4326)
    return pd.DataFrame({'lat': centroids.y, 'lon': centroids.x})


def h3_centroids(cells):
    latlng = pd.Series(cells, index=cells).map(h3.cell_to_latlng)
    return pd.DataFrame({
        'lat': latlng.map(lambda x: x[0]),
        'lon': latlng.map(lambda x: x[1]),
    })


# spatial_level -> DataFrame indexed by the unit id, with 'lat' and 'lon'
unit_centroids = {
    'h3_8':      h3_centroids(all_census_tracts['h3_index_8'].dropna().unique()),
    'h3_7':      h3_centroids(all_census_tracts['h3_index_7'].dropna().unique()),
    'census':    polygon_centroids(all_census_comm, 'geoid10'),
    'community': polygon_centroids(all_census_comm, 'commarea'),
}

for level, cen in unit_centroids.items():
    print(f'{level:10s} {len(cen):>4} centroids   '
          f'lat [{cen["lat"].min():.4f}, {cen["lat"].max():.4f}]   '
          f'lon [{cen["lon"].min():.4f}, {cen["lon"].max():.4f}]')

h3_8        849 centroids   lat [41.6447, 42.0199]   lon [-87.9374, -87.5244]
h3_7        153 centroids   lat [41.6405, 42.0264]   lon [-87.9422, -87.5149]
census      801 centroids   lat [41.6502, 42.0213]   lon [-87.9045, -87.5305]
community    77 centroids   lat [41.6601, 42.0093]   lon [-87.8882, -87.5343]


## Temporal Ground truth

In [6]:
# Pickup Hour / Dropoff Hour are parsed at load time (see the loading cell above).

# Every grid row needs weather, but the ERA5 series from 1.2 ends before the taxi data does. Left
# alone, the uncovered tail would get a forward-filled, frozen weather reading -- and because the
# split is temporal, that tail lands entirely in the TEST set. Cut the taxi data at the last hour the
# weather actually covers instead, so no row is ever scored against invented weather.
weather_end = pd.read_parquet('../data/weather_data.parquet', columns=['time_step'])['time_step'].max()

before = len(taxi_data_census), len(taxi_data_ca)

# A trip is kept only if BOTH its pickup and its dropoff fall inside the covered window; otherwise a
# trip starting just before the cutoff would have its dropoff silently dropped from Total_Trip_End.
taxi_data_census = taxi_data_census[
    (taxi_data_census['Pickup Hour'] <= weather_end) & (taxi_data_census['Dropoff Hour'] <= weather_end)
]
taxi_data_ca = taxi_data_ca[
    (taxi_data_ca['Pickup Hour'] <= weather_end) & (taxi_data_ca['Dropoff Hour'] <= weather_end)
]

print(f"weather covers through {weather_end}")
print(f"census set:    {before[0]:>10,} -> {len(taxi_data_census):>10,} trips "
      f"({before[0] - len(taxi_data_census):,} beyond the weather window)")
print(f"community set: {before[1]:>10,} -> {len(taxi_data_ca):>10,} trips "
      f"({before[1] - len(taxi_data_ca):,} beyond the weather window)")

# Span the timestamp range across BOTH datasets so the grid never truncates the community
# aggregation, and end it exactly at the weather cutoff.
min_timestamp = min(taxi_data_census['Pickup Hour'].min(), taxi_data_ca['Pickup Hour'].min())
max_timestamp = weather_end

# Create a complete hourly timestamp range and a daily timestamp range for the entire dataset
hourly_timestamps = pd.date_range(start=min_timestamp, end=max_timestamp, freq='h')
daily_timestamps = pd.date_range(start=min_timestamp, end=max_timestamp, freq='D')
print(f"\ngrid spans {min_timestamp} -> {max_timestamp}  "
      f"({len(hourly_timestamps):,} hours / {len(daily_timestamps):,} days)")

weather covers through 2026-05-12 23:00:00
census set:     6,835,451 ->  6,636,374 trips (199,077 beyond the weather window)
community set: 14,494,804 -> 14,108,693 trips (386,111 beyond the weather window)

grid spans 2024-01-01 00:00:00 -> 2026-05-12 23:00:00  (20,712 hours / 863 days)


## Spatial Temaporal Discretization
Now we use the Spatial Ground Truth to aggregate hour data on different spatial and temporal granuality levels.

In [7]:
def most_common_company(x):
    # mode() is the slowest step of the whole aggregation, so call it once per group, not twice.
    modes = x.mode()
    return modes.iloc[0] if len(modes) else np.nan


# Define aggregations specifically for the Pickups.
# 'Trip ID' is never null, so group size is the trip count and we can skip loading the column.
pickup_aggregations = {
    'Total_Trip_Start': ('Taxi ID', 'size'),
    'Unique Taxis': ('Taxi ID', 'nunique'),
    'AvgTripSeconds': ('Trip Seconds', 'mean'),
    'AvgTripMiles': ('Trip Miles', 'mean'),
    'AvgFare': ('Trip Total', 'mean'),
    'MostCommonCompany': ('Company', most_common_company),
    'CompanyCount': ('Company', 'nunique'),
    # TODO: Mean might not be the best way to aggregate lat/lon values, consider using the centroid of the H3 cell instead
    'PickupLongitude': ('Pickup Centroid Longitude', 'mean'),
    'PickupLatitude': ('Pickup Centroid Latitude', 'mean'), 
}

# Define aggregations specifically for the Dropoffs
dropoff_aggregations = {
    'Total_Trip_End': ('Taxi ID', 'size')
}

In [8]:
# The census/community codes are float64 in the taxi data (e.g. 8.0) but strings in the
# gpkg-derived grid (e.g. '8'). Normalize them to matching string codes so the merge hits.
# Each dataset only needs the key for the level(s) it feeds:
#   taxi_data_census -> census level (tract key); h3 levels use the native h3 columns
#   taxi_data_ca     -> community level (community-area key)
taxi_data_census['pickup_tract']  = taxi_data_census['Pickup Census Tract'].astype('Int64').astype('str')
taxi_data_census['dropoff_tract'] = taxi_data_census['Dropoff Census Tract'].astype('Int64').astype('str')

taxi_data_ca['pickup_comm']  = taxi_data_ca['Pickup Community Area'].astype('Int64').astype('str')
taxi_data_ca['dropoff_comm'] = taxi_data_ca['Dropoff Community Area'].astype('Int64').astype('str')

# Temporal keys floored to the day, for the daily aggregation level.
# (Hour-floored keys would only match the midnight hour of a day-floored grid.)
for _df in (taxi_data_census, taxi_data_ca):
    _df['Pickup Day']  = _df['Pickup Hour'].dt.floor('D')
    _df['Dropoff Day'] = _df['Dropoff Hour'].dt.floor('D')

In [9]:
# Map each spatial level to its pickup column, dropoff column, and the grid/universe column.
# Pickups and dropoffs live in SEPARATE columns, so each side is grouped on its own key.
spatial_map = {
    'h3_8':      {'pickup': 'h3_index_pickup_8',    'dropoff': 'h3_index_dropoff_8',    'grid': 'h3_index_8'},
    'h3_7':      {'pickup': 'h3_index_pickup_7',    'dropoff': 'h3_index_dropoff_7',    'grid': 'h3_index_7'},
    'census':    {'pickup': 'pickup_tract',         'dropoff': 'dropoff_tract',         'grid': 'geoid10'},
    'community': {'pickup': 'pickup_comm',          'dropoff': 'dropoff_comm',          'grid': 'commarea'},
}

# The complete spatial universe each level is merged against.
#
# The h3 levels come from the hexagon lookup, but tracts and community areas have to come from the
# boundary file directly. all_census_tracts holds one row per res-8 hexagon labelled with the tract
# containing that hexagon's CENTROID, so using it as the tract universe quietly shrinks the universe
# to "tracts that happen to contain a hexagon centroid". Around 70% of Chicago's tracts are smaller
# than a res-8 hexagon (0.74 km2), so 267 of the 801 tracts contain no centroid, never appear in
# complete_grid, and the left merge below drops every trip picked up in them -- 1.2M trips, 18% of
# the census set, concentrated in the small dense downtown tracts.
spatial_universe = {
    'h3_8':      all_census_tracts['h3_index_8'],
    'h3_7':      all_census_tracts['h3_index_7'],
    'census':    all_census_comm['geoid10'],
    'community': all_census_comm['commarea'],
}

temporal_map = {
    'hourly': {'range': hourly_timestamps, 'pickup': 'Pickup Hour', 'dropoff': 'Dropoff Hour'},
    'daily':  {'range': daily_timestamps,  'pickup': 'Pickup Day',  'dropoff': 'Dropoff Day'},
}

count_columns = ['Total_Trip_Start', 'Unique Taxis', 'CompanyCount', 'Total_Trip_End']

# Keep every grid in memory so the reduction can be inspected before anything is written to disk.
# Keyed by (spatial_level, temporal_level).
grids_full = {}      # the complete Chicago universe, every spatial unit
grids_reduced = {}   # only units that see at least one pickup in the whole timeframe
active_units = {}    # spatial_level -> set of units that ever see demand

for spatial_level, s in spatial_map.items():
    for temporal_level, t in temporal_map.items():
        print(f"Processing {spatial_level} spatial aggregation with {temporal_level} temporal aggregation...")
        grid_col = s['grid']
        key = (spatial_level, temporal_level)

        # Community area is populated for every trip in the big set (taxi_data_ca), so use it
        # there for maximum coverage. Every other level uses the smaller census set: census
        # needs the tract label (absent for ~53% of big-set rows), and the h3 indices are only
        # accurate on the census set -- in the big set, trips without a tract fall back to the
        # community-area centroid, which would snap their h3 cells to the CA centre.
        if spatial_level == 'community':
            taxi_data = taxi_data_ca
        else:
            taxi_data = taxi_data_census

        # Complete spatial universe of Chicago (covers every cell/tract/area, including ones with
        # no trips in the current split).
        unique_spatial = spatial_universe[spatial_level].dropna().drop_duplicates()
        complete_grid = pd.MultiIndex.from_product(
            [t['range'], unique_spatial], names=['timestamp', grid_col]
        ).to_frame(index=False)

        # Aggregate pickups on the pickup key, dropoffs on the dropoff key
        aggregated_pickups = taxi_data.groupby(
            [t['pickup'], s['pickup']]
        ).agg(**pickup_aggregations).reset_index()
        aggregated_dropoffs = taxi_data.groupby(
            [t['dropoff'], s['dropoff']]
        ).agg(**dropoff_aggregations).reset_index()

        # Merge pickups: drop only the redundant time column, KEEP the spatial key for the next merge
        complete_grid = complete_grid.merge(
            aggregated_pickups,
            left_on=['timestamp', grid_col],
            right_on=[t['pickup'], s['pickup']],
            how='left',
        ).drop(columns=[t['pickup']])
        if s['pickup'] != grid_col:
            complete_grid = complete_grid.drop(columns=[s['pickup']])

        # Merge dropoffs the same way
        complete_grid = complete_grid.merge(
            aggregated_dropoffs,
            left_on=['timestamp', grid_col],
            right_on=[t['dropoff'], s['dropoff']],
            how='left',
        ).drop(columns=[t['dropoff']])
        if s['dropoff'] != grid_col:
            complete_grid = complete_grid.drop(columns=[s['dropoff']])

        # Every pickup should land somewhere in the grid. If any don't, the universe above is missing
        # spatial units that the taxi data references, and those trips have just been thrown away.
        # A handful of h3 pickups sit just outside the city polygon, hence the tolerance.
        matched = complete_grid['Total_Trip_Start'].sum()
        expected = taxi_data[s['pickup']].notna().sum()
        if (expected - matched) / expected > 0.0001:
            print(f"  WARNING: {expected - matched:,.0f} of {expected:,} pickups "
                  f"({(1 - matched / expected) * 100:.2f}%) had no matching {grid_col} in the grid "
                  f"and were dropped.")

        # Counts of empty cells are true zeros; the mode-company gets an empty string.
        # MostCommonCompany inherits Company's categorical dtype from the aggregation, and "" is
        # not one of its categories, so step out of the Categorical before filling.
        complete_grid[count_columns] = complete_grid[count_columns].fillna(0)
        complete_grid['MostCommonCompany'] = (
            complete_grid['MostCommonCompany'].astype(object).fillna("")
        )

        # Attach the unit's geometric centre to every row (defined even for empty cells) and use it
        # to fill the pickup coordinates wherever no trip was averaged. This is a structural fill,
        # not an estimate: it says "this cell sits here", which holds regardless of demand.
        cen = unit_centroids[spatial_level]
        complete_grid['lat'] = complete_grid[grid_col].map(cen['lat'])
        complete_grid['lon'] = complete_grid[grid_col].map(cen['lon'])
        complete_grid['PickupLatitude'] = complete_grid['PickupLatitude'].fillna(complete_grid['lat'])
        complete_grid['PickupLongitude'] = complete_grid['PickupLongitude'].fillna(complete_grid['lon'])

        # AvgTripSeconds / AvgTripMiles / AvgFare are deliberately LEFT as NaN. A cell with no trips
        # has no average fare, and any constant fill would invent one. A per-cell or global mean
        # computed here would also be computed over the WHOLE timeframe, leaking test-period
        # information into the training rows -- so that imputation belongs in feature engineering,
        # fitted on the training split alone. See the NaN report below.

        # A unit that never sees a single pickup across the whole timeframe carries no signal -- it
        # contributes nothing but structural zeros (408 of the 849 res-8 hexes are like this, i.e.
        # 8.6M all-zero rows in the hourly grid). Derive the active set from the pickup aggregation
        # rather than from this grid, so that the hourly and daily levels reduce to the SAME units.
        if spatial_level not in active_units:
            demand = aggregated_pickups.groupby(s['pickup'], observed=True)['Total_Trip_Start'].sum()
            active_units[spatial_level] = set(demand[demand > 0].index) & set(unique_spatial)

        reduced = complete_grid[
            complete_grid[grid_col].isin(active_units[spatial_level])
        ].reset_index(drop=True)

        grids_full[key] = complete_grid
        grids_reduced[key] = reduced

        print(f"  full: {len(complete_grid):>10,} rows / {complete_grid[grid_col].nunique():>4} units"
              f"   reduced: {len(reduced):>10,} rows / {reduced[grid_col].nunique():>4} units")

print("\nAll spatial and temporal aggregation levels processed. Nothing written to disk yet.")

Processing h3_8 spatial aggregation with hourly temporal aggregation...


  full: 17,584,488 rows /  849 units   reduced:  9,133,992 rows /  441 units
Processing h3_8 spatial aggregation with daily temporal aggregation...


  full:    732,687 rows /  849 units   reduced:    380,583 rows /  441 units
Processing h3_7 spatial aggregation with hourly temporal aggregation...


  full:  3,168,936 rows /  153 units   reduced:  2,464,728 rows /  119 units
Processing h3_7 spatial aggregation with daily temporal aggregation...


  full:    132,039 rows /  153 units   reduced:    102,697 rows /  119 units
Processing census spatial aggregation with hourly temporal aggregation...


  full: 16,590,312 rows /  801 units   reduced: 12,427,200 rows /  600 units
Processing census spatial aggregation with daily temporal aggregation...


  full:    691,263 rows /  801 units   reduced:    517,800 rows /  600 units
Processing community spatial aggregation with hourly temporal aggregation...


  full:  1,594,824 rows /   77 units   reduced:  1,594,824 rows /   77 units
Processing community spatial aggregation with daily temporal aggregation...


  full:     66,451 rows /   77 units   reduced:     66,451 rows /   77 units

All spatial and temporal aggregation levels processed. Nothing written to disk yet.


## Reduction to active spatial units

Roughly half the res-8 hexagons never see a single pickup in the whole 2.4-year window, so they add
nothing but structural zeros to the grid. Below we compare each grid before and after dropping the
units with no demand — every trip is retained, only all-zero rows disappear.

In [10]:
def grid_stats(df, grid_col):
    """Descriptive stats for one grid, used to compare the full and reduced versions."""
    demand = df['Total_Trip_Start']
    return {
        'units': df[grid_col].nunique(),
        'rows': len(df),
        'trips': int(demand.sum()),
        'zero_rows_%': (demand == 0).mean() * 100,
        'nan_avgfare_%': df['AvgFare'].isna().mean() * 100,
        'mean_demand': demand.mean(),
        'median_demand': demand.median(),
        'std_demand': demand.std(),
        'max_demand': demand.max(),
    }


rows = []
for (spatial_level, temporal_level), full in grids_full.items():
    grid_col = spatial_map[spatial_level]['grid']
    reduced = grids_reduced[(spatial_level, temporal_level)]

    before = grid_stats(full, grid_col)
    after = grid_stats(reduced, grid_col)
    rows.append({
        'grid': f'{spatial_level}_{temporal_level}',
        **{f'{k}_before': v for k, v in before.items()},
        **{f'{k}_after': v for k, v in after.items()},
        'units_dropped': before['units'] - after['units'],
        'rows_dropped_%': (1 - after['rows'] / before['rows']) * 100,
        'trips_lost': before['trips'] - after['trips'],
    })

reduction_stats = pd.DataFrame(rows).set_index('grid')

# The headline comparison: how much of each grid was dead weight, and confirmation that dropping it
# costs zero trips.
overview = reduction_stats[[
    'units_before', 'units_after', 'units_dropped',
    'rows_before', 'rows_after', 'rows_dropped_%',
    'trips_before', 'trips_lost',
]]
print('SIZE OF THE REDUCTION')
display(overview.style.format({
    'units_before': '{:,.0f}', 'units_after': '{:,.0f}', 'units_dropped': '{:,.0f}',
    'rows_before': '{:,.0f}', 'rows_after': '{:,.0f}', 'rows_dropped_%': '{:.1f}%',
    'trips_before': '{:,.0f}', 'trips_lost': '{:,.0f}',
}))

# How the demand distribution changes once the structural zeros are gone.
distribution = reduction_stats[[
    'zero_rows_%_before', 'zero_rows_%_after',
    'nan_avgfare_%_before', 'nan_avgfare_%_after',
    'mean_demand_before', 'mean_demand_after',
    'std_demand_before', 'std_demand_after',
    'max_demand_before',
]]
print('\nEFFECT ON THE DEMAND DISTRIBUTION')
display(distribution.style.format({
    'zero_rows_%_before': '{:.1f}%', 'zero_rows_%_after': '{:.1f}%',
    'nan_avgfare_%_before': '{:.1f}%', 'nan_avgfare_%_after': '{:.1f}%',
    'mean_demand_before': '{:.2f}', 'mean_demand_after': '{:.2f}',
    'std_demand_before': '{:.2f}', 'std_demand_after': '{:.2f}',
    'max_demand_before': '{:,.0f}',
}))

assert (reduction_stats['trips_lost'] == 0).all(), 'reduction dropped trips -- it must not'
print('\nOK: every trip is retained at every level; only all-zero rows were removed.')

SIZE OF THE REDUCTION


,units_before,units_after,units_dropped,rows_before,rows_after,rows_dropped_%,trips_before,trips_lost
grid,,,,,,,,
h3_8_hourly,849,441,408,"17,584,488","9,133,992",48.1%,"6,636,365",0
h3_8_daily,849,441,408,"732,687","380,583",48.1%,"6,636,365",0
h3_7_hourly,153,119,34,"3,168,936","2,464,728",22.2%,"6,636,357",0
h3_7_daily,153,119,34,"132,039","102,697",22.2%,"6,636,357",0
census_hourly,801,600,201,"16,590,312","12,427,200",25.1%,"6,636,374",0
census_daily,801,600,201,"691,263","517,800",25.1%,"6,636,374",0
community_hourly,77,77,0,"1,594,824","1,594,824",0.0%,"14,108,693",0
community_daily,77,77,0,"66,451","66,451",0.0%,"14,108,693",0



EFFECT ON THE DEMAND DISTRIBUTION


,zero_rows_%_before,zero_rows_%_after,nan_avgfare_%_before,nan_avgfare_%_after,mean_demand_before,mean_demand_after,std_demand_before,std_demand_after,max_demand_before
grid,,,,,,,,,
h3_8_hourly,98.2%,96.6%,98.2%,96.6%,0.38,0.73,5.18,7.18,381
h3_8_daily,93.9%,88.3%,93.9%,88.3%,9.06,17.44,92.84,128.25,"3,557"
h3_7_hourly,94.7%,93.2%,94.7%,93.2%,2.09,2.69,18.05,20.42,527
h3_7_daily,83.6%,78.9%,83.6%,78.9%,50.26,64.62,333.88,377.36,"5,582"
census_hourly,97.6%,96.7%,97.6%,96.7%,0.40,0.53,4.75,5.48,334
census_daily,91.6%,88.8%,91.6%,88.8%,9.60,12.82,85.44,98.51,"3,557"
community_hourly,43.2%,43.2%,43.2%,43.2%,8.85,8.85,35.35,35.35,566
community_daily,0.7%,0.7%,0.7%,0.7%,212.32,212.32,684.10,684.10,"6,586"



OK: every trip is retained at every level; only all-zero rows were removed.


In [11]:
def generate_nan_report(df):
    """One row per column that still contains NaN, with the share of rows affected."""
    counts = df.isna().sum()
    counts = counts[counts > 0]
    report = pd.DataFrame({'NaN Count': counts, 'Total Count': len(df)})
    report['NaN Percentage'] = (report['NaN Count'] / report['Total Count'] * 100).map('{:.2f}%'.format)
    return report


print('NaN report for the grids that get written (reduced):\n')
for (spatial_level, temporal_level), df in grids_reduced.items():
    name = f'{spatial_level}_{temporal_level}'
    report = generate_nan_report(df)
    if report.empty:
        print(f'{name}: no NaN values.')
    else:
        print(f'{name}:')
        print(report.to_string())
    print(40 * '-')

# The coordinate columns must be complete after the centroid fill -- an empty cell still has a
# location, so a NaN here would mean a unit is missing from unit_centroids.
for key, df in grids_reduced.items():
    for col in ('lat', 'lon', 'PickupLatitude', 'PickupLongitude'):
        assert df[col].notna().all(), f'{col} still has NaN in {key}'
print('\nOK: lat/lon and PickupLatitude/PickupLongitude are complete at every level.')

# What remains is AvgTripSeconds / AvgTripMiles / AvgFare on the zero-demand rows. These are left
# NaN on purpose: there is no average of no trips. Filling them here with a cell or global mean
# would (a) invent a value and (b) compute it across the whole timeframe, leaking the test period
# into training. Impute them downstream from training-split statistics, or let the model consume
# them alongside Total_Trip_Start == 0, which already encodes "nothing happened here".
remaining = sorted({c for df in grids_reduced.values() for c in generate_nan_report(df).index})
print(f'Columns still holding NaN by design: {remaining}')

NaN report for the grids that get written (reduced):



h3_8_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds    8824834      9133992         96.62%
AvgTripMiles      8824834      9133992         96.62%
AvgFare           8824834      9133992         96.62%
----------------------------------------
h3_8_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     336134       380583         88.32%
AvgTripMiles       336134       380583         88.32%
AvgFare            336134       380583         88.32%
----------------------------------------
h3_7_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds    2298278      2464728         93.25%
AvgTripMiles      2298278      2464728         93.25%
AvgFare           2298278      2464728         93.25%
----------------------------------------
h3_7_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds      81058       102697         78.93%
AvgTripMiles        81058       102697         78.93%
AvgFare          

census_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds   12022051     12427200         96.74%
AvgTripMiles     12022051     12427200         96.74%
AvgFare          12022051     12427200         96.74%
----------------------------------------
census_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     460058       517800         88.85%
AvgTripMiles       460058       517800         88.85%
AvgFare            460058       517800         88.85%
----------------------------------------
community_hourly:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds     688356      1594824         43.16%
AvgTripMiles       688356      1594824         43.16%
AvgFare            688356      1594824         43.16%
----------------------------------------
community_daily:
                NaN Count  Total Count NaN Percentage
AvgTripSeconds        474        66451          0.71%
AvgTripMiles          474        66451          0.71%
Avg

Columns still holding NaN by design: ['AvgFare', 'AvgTripMiles', 'AvgTripSeconds']


In [12]:
# Write the reduced grids -- these are what the feature engineering and models consume.
# The full grids stay in `grids_full` for anyone who needs the complete Chicago universe (e.g. maps
# that should still show the empty parts of the city).
for (spatial_level, temporal_level), df in grids_reduced.items():
    path = f'../data/processed/complete_grid_{spatial_level}_{temporal_level}.parquet'
    df.to_parquet(path, index=False)
    print(f'wrote {path}  ({len(df):,} rows)')

print('\nAll reduced grids written.')

wrote ../data/processed/complete_grid_h3_8_hourly.parquet  (9,133,992 rows)
wrote ../data/processed/complete_grid_h3_8_daily.parquet  (380,583 rows)


wrote ../data/processed/complete_grid_h3_7_hourly.parquet  (2,464,728 rows)
wrote ../data/processed/complete_grid_h3_7_daily.parquet  (102,697 rows)


wrote ../data/processed/complete_grid_census_hourly.parquet  (12,427,200 rows)
wrote ../data/processed/complete_grid_census_daily.parquet  (517,800 rows)


wrote ../data/processed/complete_grid_community_hourly.parquet  (1,594,824 rows)
wrote ../data/processed/complete_grid_community_daily.parquet  (66,451 rows)

All reduced grids written.
